# Notebook 1 — Data Gathering and Manipulation (NumPy & Pandas)

**Duration:** 75 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** locate, load, inspect, clean, and perform basic feature engineering on the Fe-Redox dataset so it is ready for analysis and model training.


## Learning objectives

- Find the dataset files in the repository and load them into pandas.
- Inspect data types, summary statistics, and identify missing values.
- Detect and handle outliers using the IQR method.
- Compute simple structural features (e.g., Fe–ligand bond lengths) from coordinate data.
- Save a cleaned CSV for downstream analysis.

---

## Part 1 — Locating and understanding the data

### 1.1 Repository layout and data location

Start by inspecting the repository to find likely data folders (examples: `data/`, `datasets/`, `raw/`, `structures/`). Use the Jupyter file browser or a short shell listing in a terminal cell.

### 1.2 Loading the main dataset

The workshop uses the tmQM redox dataset distributed with the companion `gnnredox/` clone. Start the kernel from the workspace root so `./gnnredox` resolves, or set `REPO_PATH` to the absolute path of your clone.



In [ ]:
# Standard imports
import os
from pathlib import Path
import pandas as pd

REPO_PATH = os.getcwd() + '/..'

if REPO_PATH is None:
   raise FileNotFoundError(
       'Could not find the gnnredox clone. Clone https://github.com/alvarovm/GraphNetwork-Redox '
       'as gnnredox/ beside this workshop, or set GNNREDOX_PATH.'
   )
print(f'Using gnnredox clone at: {REPO_PATH}')

Using gnnredox clone at: /home/vama/soft/hpcbootcamp2026/GNNrepo/notebooks/..


In [7]:

# All data containing  complexes and redox potentials
DATA_PATH = REPO_PATH +'/data/tmqm_redox_data_full_data.csv'
df = pd.read_csv(DATA_PATH)
df = df.set_index('csd_code')

print(f'Loaded {len(df)} complexes from: {DATA_PATH}')

# Mol and smiles of complexes generated directly from tmQM dataset
# This also represents the full redox dataset
DATA_PATH = REPO_PATH + '/data/tmc_frm_xyz2mol_tmqm_all_df.pkl'
tmqm_final_df = pd.read_pickle(DATA_PATH)
tmqm_final_df = tmqm_final_df.drop_duplicates(subset='csd_code', keep='first')
tmqm_final_df = tmqm_final_df.set_index('csd_code')

# # Mol and smiles of complexes generated after semi-impirically optimizing the geometries solvated exlicitly with water and then desolvating
# # This also represents the gnn-redox dataset
DATA_PATH = REPO_PATH + '/data/final_df.pkl'
final_df = pd.read_pickle(DATA_PATH)

Loaded 2267 complexes from: /home/vama/soft/hpcbootcamp2026/GNNrepo/notebooks/../data/tmqm_redox_data_full_data.csv


In [ ]:
tmqm_final_df

### Filters for the gnn-redox dataset

In [ ]:
print(f'Columns in df = {df.columns.tolist()}') 
print(f'    Shape of DF = {df.shape}')
print(f'-'*40)

# final_df.drop(columns=["y"], inplace=True)
print(f'Columns in final_df = {final_df.columns.tolist()}') 
print(f'    Shape of final_df = {final_df.shape}')
print(f'-'*40)
# tmqm_final_df

# tmqm_final_df.drop(columns=["y"], inplace=True)
print(f'Columns in tmqm_final_df = {tmqm_final_df.columns.tolist()}') 
print(f'    Shape of tmqm_final_df = {tmqm_final_df.shape}')


### Merge Dataframes

In [ ]:
df

In [ ]:

df = pd.concat([df, final_df], axis=1, join="inner")


df = pd.concat([df, tmqm_final_df], axis=1, join="inner")

print(f'New Columns in df = {df.columns.tolist()}') 
print(f' New Shape of DF = {df.shape}')


### 1.3 Alternative structural parsing (XYZ-like files)

The structure are represented as ASE atoms objects. The objects are stored in tables in Pickle format, which allows to serialize the python objects.



In [ ]:
# import numpy as np

# df1= pd.read_pickle('/home/vama/soft/hpcbootcamp2026/gnnredox/Data/desolvated_tmqm_all_xyz.pkl')
# df1 = df1.set_index('csd_code')

# df = pd.concat([df, df1], axis=1)

def parse_distances(atomx):
    # atomx = df1.tmqm_atoms[1]
    for k,v in enumerate(atomx):
        if v.symbol == 'Fe':
            fe_atom = k
            break

    distances = atomx.get_all_distances()[fe_atom]
    distances = np.delete(distances,fe_atom)
    return distances



---

## Part 2 — Data inspection with pandas

### 2.1 First look

Show the first few rows and the shape of the data. Use `display()` in Jupyter so wide tables render nicely.



In [ ]:
from IPython.display import display

if 'df' not in globals():
    raise RuntimeError("Load the dataset in the previous cell before running this one.")

print("Shape:", df.shape)
display(df.head())



### 2.2 Data types and summary statistics

Inspect dtypes and a numeric summary. This helps identify columns that need casting or contain unexpected values.



In [ ]:
print(df.dtypes)
display(df.describe(include='all').T)



### 2.3 Column completeness and uniqueness

Check non-null counts and unique values for categorical columns. This gives a quick picture of missingness and label cardinality.



In [ ]:
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls>0])

cat_cols = df.select_dtypes(include=['object','category' ]).columns.tolist()

cat_cols.remove('desolv_atoms')
cat_cols.remove('tmqm_atoms')
cat_cols.remove('atoms')
cat_cols
for c in cat_cols:
    print(c, "-> unique:", df[c].nunique())



### 2.4 Physical meaning of tmQM columns

| Column | Meaning | Notes |
|---|---:|---|
| `csd_code` | Cambridge Structural Database identifier | unique complex identifier |
| `q` | Overall complex charge | integer |
| `Stoichiometry` | Elemental composition | compact formula, e.g. `C12H14FeN4O10` |
| `num_atoms` | Number of atoms | integer |
| `ligands_list` | Serialized ligand SMILES list | supports ligand-count features |
| `tm_oxs` | Transition-metal oxidation state | constant for this Fe(II) subset |
| `ligands_q_list` | Serialized ligand formal charges | supports charge-derived features |
| `reduction_pot` | Experimental reduction potential | regression target in volts |

### 2.5 Feature engineering from composition and ligands

The raw table has only two useful native numeric predictors (`q` and `num_atoms`); `tm_oxs` is constant. Derive composition and ligand features so PCA and the baseline models have meaningful inputs.



In [ ]:
import ast
import re

TARGET_COL = 'reduction_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Expected target column '{TARGET_COL}' was not found")

def parse_stoichiometry(formula):
    return {
        element: int(count) if count else 1
        for element, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    }

stoichiometry = df['Stoichiometry'].map(parse_stoichiometry)
for element in ['C', 'H', 'N', 'O', 'S', 'Cl', 'P']:
    df[f'n_{element}'] = stoichiometry.map(lambda counts, el=element: counts.get(el, 0))

df['n_ligands'] = df['ligands_list'].map(lambda value: len(ast.literal_eval(value)))
df['q_sum'] = df['ligands_q_list'].map(lambda value: sum(ast.literal_eval(value)))
df['n_anionic'] = df['ligands_q_list'].map(
    lambda value: sum(charge < 0 for charge in ast.literal_eval(value))
)

# constant_cols = [c for c in df.select_dtypes(include='number') if df[c].nunique() <= 1]
# if constant_cols:
#     print('Dropping constant columns:', constant_cols)
#     df = df.drop(columns=constant_cols)

numeric_cols = df.select_dtypes(include='number').columns.tolist()
FEATURE_COLS = [c for c in numeric_cols if c != TARGET_COL]
print('Target:', TARGET_COL)
print('Numeric features:', FEATURE_COLS)



---

## Part 3 — Data cleaning

### 3.1 Handling missing values

Strategy: prefer dropping rows with missing target values; for features, either impute (mean/median) or flag missingness via an indicator column.



In [ ]:
# Drop rows missing the target
before = len(df)
df = df[df[TARGET_COL].notnull()].copy()
print(f"Dropped {before - len(df)} rows with missing target '{TARGET_COL}'")

# Impute numeric features with median to be robust to outliers
num_cols = [c for c in df.select_dtypes(include='number') if c != TARGET_COL]
for c in num_cols:
    if df[c].isnull().any():
        med = df[c].median()
        df[f"{c}_was_missing"] = df[c].isnull()
        df[c] = df[c].fillna(med)
        print(f"Imputed {c} with median = {med}")



### 3.2 Outlier detection with the IQR method

We use the interquartile range (IQR) to flag extreme values. Recall the definitions:

IQR and outlier bounds:

$$
\text{IQR} = Q_3 - Q_1
$$

A sample $x$ is considered an outlier if:

$$
x < Q_1 - 1.5\,\text{IQR} \quad \text{or} \quad x > Q_3 + 1.5\,\text{IQR}
$$

Apply this per numeric column and optionally remove the flagged rows or save a Boolean mask.



In [ ]:
import numpy as np

outlier_mask = pd.DataFrame(False, index=df.index, columns=num_cols)

for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[c] < lower) | (df[c] > upper)
    outlier_mask[c] = mask
    print(f"{c}: outliers {mask.sum()} (lower={lower}, upper={upper})")

# Aggregate outlier flag (True if any feature flags an outlier)
df['is_outlier_any'] = outlier_mask.any(axis=1)
print('Total rows with any outlier:', df['is_outlier_any'].sum())

# Optional: remove extreme outliers (commented by default)
# df = df[~df['is_outlier_any']].copy()



Produce a simple boxplot figure of selected columns to visualize outliers.



In [ ]:
import matplotlib.pyplot as plt

sel = [c for c in num_cols if c in ['n_O', 'n_ligands', 'n_anionic', 'num_atoms', 'q']][:6]
if not sel:
    sel = num_cols[:6]

fig, axes = plt.subplots(1, len(sel), figsize=(4*len(sel), 4), squeeze=False)
axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, sel):
    ax.boxplot(df[c].dropna())
    ax.set_title(c)
# Hide unused axes
for ax in axes[len(sel):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig('outlier_boxplots.png', dpi=150)
plt.show()



### 3.3 Feature engineering: compute Fe–ligand bond lengths

We compute Euclidean distances between Fe coordinates and ligand atom coordinates. In 3D, the Euclidean distance is:

$$
d_{ij} = \lVert \mathbf{r}_i - \mathbf{r}_j \rVert_2 = \sqrt{\sum_{k=1}^{3} (r_{i,k} - r_{j,k})^2}
$$

Implement a helper that extracts Fe coordinates from a row and computes bond-length statistics (min, max, mean) to be used as features.



In [ ]:
import math


def compute_fe_bond_lengths(row):
    atomx = row.desolv_atoms
    dists = parse_distances(atomx)
    return pd.Series({'fe_bond_min':  float(np.nanmin(dists)),
                      'fe_bond_mean': float(np.nanmean(dists)),
                      'fe_bond_max':  float(np.nanmax(dists))})


# Apply to the dataframe (may be slow; consider vectorized approaches for large datasets)
# if 'fe_x' in df.columns:
fe_features = df.apply(compute_fe_bond_lengths, axis=1)
df = pd.concat([df, fe_features], axis=1)
print('Computed Fe bond-length features')
# else:
#     print('fe_x/fe_y/fe_z not found; skipping Fe bond-length computation')



---

## Part 4 — Data filtering and querying

Demonstrate typical data filters: restrict to a single oxidation state, remove very large complexes, or restrict by computed criteria.



In [ ]:
# Example: select Fe(2) complexes only
if 'fe_oxidation_state' in df.columns:
    df = df[df['fe_oxidation_state'] == 2].copy()
    print('Filtered to Fe(2). Remaining rows:', len(df))

# Example: remove extreme charge values
if 'charge' in df.columns:
    df = df[df['charge'].abs() <= 3].copy()
    print('Removed large charges. Remaining rows:', len(df))



**Mentor checkpoint 3**

- Confirm dataset selection and target column are correct for the group.
- Discuss any domain-specific filters (oxidation state, solvent, basis set) to apply before modeling.
- Decide whether to drop outliers flagged by `is_outlier_any` or to keep them and add robust models later.

Proceed only after confirmation.

---

### Exercise 1.1 — Quick data exploration (10 minutes)

1. Print the distribution (value counts) of a categorical column such as `ligand_type` or `solvent` if present.
2. Report the number of unique Fe centers (hint: group by a unique complex identifier).
3. Compute the fraction of rows that were flagged as outliers.

Write code to answer these questions and print short conclusions (1-2 lines each).

### Exercise 1.2 — NumPy operations on structure arrays (20 minutes)

1. Using the `parse_xyz_file` helper, read one structure file and compute the centroid of its atoms. Use NumPy vector operations.

Recall z-score normalization for a sample:

$$
z_i = \frac{x_i - \mu}{\sigma}
$$

and the sample covariance between X and Y:

$$
\mathrm{Cov}(X,Y) = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})
$$

2. Compute the z-scores for `n_O` and compute its covariance with `reduction_pot`.

### Exercise 1.3 — Save the cleaned dataset (5 minutes)

Save `df` to `data_cleaned.csv` in the notebook folder. This file will be used by Notebook 2.



In [ ]:
# OUTPUT_DIR = Path(os.environ.get('SCRATCH', '.')) / 'Fe-Redox-GNN'
thispath = os.getcwd()
OUTPUT_DIR =  thispath + '/../output'
if not os.path.isdir(OUTPUT_DIR):
    os.mkdir(OUTPUT_DIR)

CLEANED_PATH = OUTPUT_DIR +'/data_cleaned.pkl'
df.to_pickle(CLEANED_PATH)#, index=False)
print('Wrote', CLEANED_PATH, 'with', len(df), 'rows')



---

## Summary

| Task | Tool | Key functions |
|---|---:|---|
| Load data | pandas | `pd.read_csv` |
| Inspect | pandas | `df.head`, `df.describe`, `df.dtypes` |
| Outlier detection | pandas, numpy | quantiles, boolean masks |
| Feature engineering | numpy, custom | `compute_fe_bond_lengths` |

---

# Notebook 2 — Exploratory Data Analysis & Visualization (Matplotlib & Seaborn)

**Duration:** 75 minutes  
**Audience:** Undergraduates with intermediate Python, introductory chemistry knowledge  
**Goal:** visualize target and feature distributions, identify relationships, and produce publication-quality diagnostic plots.

## Learning objectives

- Plot distributions and summary statistics of the target variable.
- Visualize pairwise relationships and compute correlation matrices (Pearson and Spearman).
- Use violin plots and 2D density plots to inspect conditional distributions.
- Save figures for use in reports and presentations.

---

## Part 1 — Distribution analysis

### 1.1 Load the cleaned data and imports



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')

# CLEANED_PATH = Path(os.environ.get('SCRATCH', '.')) / 'Fe-Redox-GNN' / 'data_cleaned.csv'
CLEANED_PATH = OUTPUT_DIR +'/data_cleaned.pkl'
df = pd.read_pickle(CLEANED_PATH)
print('Loaded cleaned data with', len(df), 'rows from', CLEANED_PATH)
display(df.head())
fe_features


### 1.2 Target distribution and moments

Plot the distribution of the redox target and compute skewness and kurtosis. The sample skewness and (excess) kurtosis are defined as:

$$
\text{skew}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{3}\right], \quad \text{kurt}(X) = \mathbb{E}\left[\left(\frac{X - \mu}{\sigma}\right)^{4}\right] - 3
$$



In [ ]:
TARGET_COL = 'reduction_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f'{TARGET_COL} column not found in cleaned data')

plt.figure(figsize=(6,4))
sns.histplot(df[TARGET_COL], kde=True)
plt.title('Reduction potential distribution')
plt.savefig('redox_distribution.png', dpi=150)
plt.show()

print('skewness:', df[TARGET_COL].skew())
print('kurtosis (excess):', df[TARGET_COL].kurt())



### 1.3 Feature distributions

Visualize a small set of numeric features with histograms and KDEs. This helps spot multimodality and heavy tails.



In [ ]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['index', TARGET_COL]]

def plot_feature_distributions(df, cols):
    n = len(cols)
    ncols = 4
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, c in zip(axes, cols):
        sns.histplot(df[c].dropna(), kde=True, ax=ax)
        ax.set_title(c)
    for ax in axes[len(cols):]:
        ax.set_visible(False)
    fig.tight_layout()
    fig.savefig('feature_distributions.png', dpi=150)
    plt.show()

plot_feature_distributions(df, num_cols[:8])



---

## Part 2 — Relationship analysis

### 2.1 Scatter plots and Pearson correlation

Pearson correlation coefficient between two variables is defined as:

$$
r_{XY} = \frac{\sum_{i} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i} (x_i - \bar{x})^2}\ \sqrt{\sum_{i} (y_i - \bar{y})^2}}
$$

Plot a selection of features against the target and overlay a linear regression fit.



In [ ]:
pairs = [p for p in ['n_O', 'n_ligands', 'n_anionic'] if p in df.columns]
if not pairs:
    pairs = num_cols[:3]

fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 4), squeeze=False)
axes = np.atleast_1d(axes).flatten()
for ax, c in zip(axes, pairs):
    sns.regplot(x=c, y=TARGET_COL, data=df, ax=ax, scatter_kws={'s':10}, line_kws={'color':'red'})
    ax.set_title(f'{c} vs {TARGET_COL}')
for ax in axes[len(pairs):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig('feature_vs_target_scatter.png', dpi=150)
plt.show()

for c in pairs:
    r = df[c].corr(df[TARGET_COL])
    print(f'Pearson r ({c} vs {TARGET_COL}):', r)



### 2.2 Pairplot for multi-feature relationships

A pairplot (scatter + KDE on diag) helps identify non-linear relationships and clusters.



In [ ]:
cols_for_pairplot = [c for c in ['n_O', 'n_ligands', 'n_anionic', 'num_atoms', 'q', TARGET_COL] if c in df.columns]
if len(cols_for_pairplot) > 1:
    sns.pairplot(df[cols_for_pairplot].dropna(), diag_kind='kde', corner=True)
    plt.savefig('pairplot.png', dpi=150)
    plt.show()



---

## Part 3 — Correlation analysis

### 3.1 Correlation matrix heatmap

Compute Pearson correlation matrix and plot a heatmap.



In [ ]:
corr = df[num_cols + [TARGET_COL]].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Pearson correlation matrix')
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()



### 3.2 Spearman rank correlation

Spearman rank correlation is useful for monotonic but non-linear relationships. The rank correlation coefficient is:

$$
\rho = 1 - \frac{6 \sum_{i} d_i^2}{n(n^2 - 1)}\quad\text{where } d_i = \mathrm{rank}(x_i) - \mathrm{rank}(y_i)
$$

Compute and compare Spearman to Pearson for the target relationships.



In [ ]:
spearman = df[num_cols + [TARGET_COL]].corr(method='spearman')
print(f'Top correlations with {TARGET_COL} (Pearson):')
print(corr[TARGET_COL].abs().sort_values(ascending=False).head(10))
print(f'\nTop correlations with {TARGET_COL} (Spearman):')
print(spearman[TARGET_COL].abs().sort_values(ascending=False).head(10))

# q and q_sum encode almost the same charge information. Keep one for modeling.
if 'q_sum' in num_cols:
    num_cols.remove('q_sum')
    print('\nDropping q_sum from model features because it duplicates q.')



---

## Part 4 — Advanced visualizations

### 4.1 Violin plots by category

If a categorical variable exists (e.g., `ligand_type` or `solvent`), violin plots show conditional distributions of the target.



In [ ]:
# cat = None
# for c in df.select_dtypes(include=['object', 'category']).columns:
#     if df[c].nunique() < 10:
#         cat = c
#         break

cat_cols = None

cat_cols = df.select_dtypes(include=['object','category' ]).columns.tolist()
cat_cols.remove('desolv_atoms')
cat_cols.remove('tmqm_atoms')
cat_cols.remove('atoms')
cat_cols
for c in cat_cols: 
     print (df[c].nunique())
     if df[c].nunique() < 10:
        cat = c
        break   

if cat:
    plt.figure(figsize=(8,4))
    sns.violinplot(x=cat, y=TARGET_COL, data=df)
    plt.xticks(rotation=45)
    plt.title(f'Redox by {cat}')
    plt.savefig('violin_plots.png', dpi=150)
    plt.show()
else:
    print('No low-cardinality categorical column found for violin plots')



### 4.2 2D density plot

A 2D kernel density estimate lets you visualize joint distributions and identify dense regions where models should focus their fit.



In [ ]:
if 'n_O' in df.columns and 'n_ligands' in df.columns:
    plt.figure(figsize=(6,5))
    sns.kdeplot(x=df['n_O'], y=df['n_ligands'], cmap='Blues', fill=True, thresh=0.05)
    plt.xlabel('n_O')
    plt.ylabel('n_l_density.png', dpi=150)
    plt.show()



**Mentor checkpoint 4**

- Confirm whether the observed correlations align with chemical intuition.
- Discuss which features are plausible inputs for classical baselines (RF, GPR) vs. GNN inputs.
- Decide on a shortlist of features to use in Day 3 baselines.

Proceed only after confirmation.

---

### Exercise 2.1 — Custom correlation heatmap (20 minutes)

Create a custom heatmap focusing on the top 10 features most correlated (absolute Pearson r) with `redox_potential`. Save it as `exercise_2_1_heatmap.png`.

Provide code that computes the top features, recomputes the correlation submatrix, and plots it with annotations.

### Exercise 2.2 — Feature relationship deep dive (25 minutes)

Pick one feature (for example `fe_bond_mean`) and explore polynomial relationships to the redox target. Fit polynomial models of degree 1..5 and report validation R² for each.

Coefficient of determination (R²):

$$
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

Hint: Split the data into a train/validation split (e.g., 80/20) or use k-fold cross-validation to compare polynomial degrees fairly and avoid overfitting.

### Exercise 2.3 — Discussion (5 minutes)

Answer in Markdown: based on the EDA, what modeling approach would you try first and why? Consider model complexity, interpretability, and dataset size.

---

## Summary

| Visualization | Purpose | Library |
|---|---:|---|
| Histogram + KDE | Inspect target distribution, skewness | seaborn |
| Scatter + regplot | Inspect linear trends with target | seaborn |
| Pairplot | Multi-feature relationships | seaborn |
| Correlation heatmap | Global linear associations | seaborn, matplotlib |
| Violin / KDE / 2D density | Conditional distributions and joint density | seaborn |

---

## Preparation for Day 3

Day 3 will cover dimensionality reduction (PCA), clustering (K-Means), and classical ML baselines (Random Forest, Gaussian Process Regression). Make sure `data_cleaned.csv` exists and confirm the shortlist of features you want to try as baselines.